In [1]:
import numpy as np
import tensorflow as tf
from typing import List, Dict, Optional, Tuple, Union

import flwr as fl


#Load the CIFAR-10 in NUM_CLIENTS different subsets for the training and test as it has been in the previous unit
import numpy as np

NUM_CLIENTS = 10

def unison_shuffled_copies(a, b):
    assert len(a) == len(b)
    p = np.random.permutation(len(a))
    return a[p], b[p]

def split_index(a, n):
    s = np.array_split(np.arange(len(a)), n)
    return s

# Code to load the dataset
def load_datasets(num_clients: int):
    # Distribute it to train and test set
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    # Normalize data
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    x_train, y_train = x_train[:10_000], y_train[:10_000]
    x_test, y_test = x_test[:1000], y_test[:1000]

    # Randomize the datasets
    x_train, y_train = unison_shuffled_copies(x_train, y_train)
    x_test, y_test = unison_shuffled_copies(x_test, y_test)

    # Split training set into 'num_clients' partitions to simulate the individual dataset
    train_index = split_index(x_train, num_clients)
    test_index = split_index(x_test, num_clients)

    # Split each partition
    train_ds = []
    val_ds = []
    test_ds = []
    for cid in range(num_clients):
        val_size = len(train_index[cid]) // 10
        train_input_data, train_output_data = x_train[train_index[cid]], y_train[train_index[cid]]
        val_input_data, val_output_data = train_input_data[:val_size], train_output_data[:val_size]
        train_input_data, train_output_data = train_input_data[val_size:], train_output_data[val_size:]
        train_dataset = (train_input_data, train_output_data)
        val_dataset = (val_input_data, val_output_data)
        test_dataset = (x_test[test_index[cid]], y_test[test_index[cid]])  
        train_ds.append(train_dataset)
        val_ds.append(val_dataset)
        test_ds.append(test_dataset)
    
    return train_ds, val_ds, test_ds


trainloaders, valloaders, testloader = load_datasets(NUM_CLIENTS)

# Define the model to be used in the clients

# # The part to adjust for each framework
# def generate_ann():
#     model = tf.keras.Sequential([
#         tf.keras.layers.Flatten(input_shape=(32, 32, 3)),
#         tf.keras.layers.Dense(64, activation='relu'),
#         tf.keras.layers.Dense(64, activation='relu'),
#         tf.keras.layers.Dense(10, activation='softmax')
#     ])

#     model.compile(
#         loss=tf.keras.losses.sparse_categorical_crossentropy,
#         optimizer=tf.keras.optimizers.Adam(),
#         metrics=['accuracy']
#     )
#     return model

# Define a new architecture for the neural network
def generate_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.Adam(),
        metrics=['accuracy']
    )
    return model


def get_parameters(net) -> List[np.array]:
    return net.get_weights()


def set_parameters(net, parameters: List[np.ndarray]):
    net.set_weights(parameters)
    return net


def train(net, trainloader, epochs: int):
    net.fit(trainloader[0], trainloader[1], epochs=epochs, batch_size=32, steps_per_epoch=3)
    return net


def test(net, testloader):
    loss, accuracy = net.evaluate(testloader[0], testloader[1])
    return loss, accuracy

# Class to contain a Client
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, cid, net, trainloader, valloader):
        self.cid = cid
        self.net = net
        self.trainloader = trainloader
        self.valloader = valloader

    def get_parameters(self, config):
        print(f"[Client {self.cid}] get_parameters")
        return get_parameters(self.net)

    def fit(self, parameters, config):
        print(f"[Client {self.cid}] fit, config: {config}")
        self.net = set_parameters(self.net, parameters)
        self.net = train(self.net, self.trainloader, epochs=1)
        return get_parameters(self.net), len(self.trainloader), {}

    def evaluate(self, parameters, config):
        print(f"[Client {self.cid}] evaluate, config: {config}")
        self.net = set_parameters(self.net, parameters)
        loss, accuracy = test(self.net, self.valloader)
        print(f"[Client {self.cid}] loss:{loss}, Client {self.cid} accuracy:{accuracy}")
        return float(loss), len(self.valloader), {"accuracy": float(accuracy)}


def client_fn(Context) -> FlowerClient:
    # Create the model
    net = generate_cnn()
    #get the identification
    partition_id = int(Context.node_config["partition-id"])
    #Take the appropiate part of the dataset
    trainloader = trainloaders[int(partition_id)]
    valloader = valloaders[int(partition_id)]
    #Create and return the Client
    return FlowerClient(partition_id, net, trainloader, valloader).to_client()

/Users/albertolandi/anaconda3/envs/ml2/lib/python3.12/site-packages/keras/src/utils/file_utils.py:115: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(


In [ ]:
from flwr.server import ServerApp, ServerAppComponents
from flwr.common import Context
from flwr.client import ClientApp

configurations = [
    {"fraction_fit": 0.1, "fraction_evaluate": 0.1},
    {"fraction_fit": 0.2, "fraction_evaluate": 0.2},
    {"fraction_fit": 0.3, "fraction_evaluate": 0.3},
    {"fraction_fit": 0.4, "fraction_evaluate": 0.4},
]

results = []

def get_evaluate_fn(input_dataset):
    # The `evaluate` function will be called by Flower after every round
    def evaluate_fn(
        server_round: int, parameters: fl.common.NDArrays, 
        config: Dict[str, fl.common.Scalar]) -> Optional[Tuple[float, Dict[str, fl.common.Scalar]]]:
        # Load the test data
        dataset = input_dataset
        
        # Update the model with the latest parameters
        model = generate_cnn()
        set_parameters(model, parameters)
    
        # Evaluate the model on the test dataset
        loss, accuracy = test(model, dataset)
    
        # Log the evaluation results
        print(f"Server-side evaluation round {server_round} with loss {loss} / accuracy {accuracy}")
        
        # Append the results with configuration details
        results.append({
            "Round": server_round,
            "Loss": loss,
            "Accuracy": accuracy
        })
        
        return loss, {"accuracy": accuracy}

    return evaluate_fn

for config in configurations:
    def server_fn(context: Context):
        model = generate_cnn()
        params = get_parameters(model)
        del model

        strategy = fl.server.strategy.FedAvg(
            fraction_fit=config["fraction_fit"],
            fraction_evaluate=config["fraction_evaluate"],
            min_fit_clients=3,
            min_evaluate_clients=2,
            min_available_clients=NUM_CLIENTS,
            initial_parameters=fl.common.ndarrays_to_parameters(params),
            evaluate_fn=get_evaluate_fn(testloader[0]),
        )

        server_config = fl.server.ServerConfig(num_rounds=2)
        return ServerAppComponents(strategy=strategy, config=server_config)

    server_app = ServerApp(server_fn=server_fn)
    client_app = ClientApp(client_fn=client_fn)

    print(f"Running simulation for config: {config}")
    fl.simulation.run_simulation(
        server_app=server_app, client_app=client_app, num_supernodes=NUM_CLIENTS
    )

INFO :      Starting Flower ServerApp, config: num_rounds=2, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters


Running simulation for config: {'fraction_fit': 0.1, 'fraction_evaluate': 0.1}
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1305 - loss: 2.3214  


INFO :      initial parameters (loss, other metrics): 2.325810432434082, {'accuracy': 0.10999999940395355}
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 2 clients (out of 10)


Server-side evaluation round 0 with loss 2.325810432434082 / accuracy 0.10999999940395355
(ClientAppActor pid=17824) [Client 9] fit, config: {}


INFO :      aggregate_fit: received 2 results and 0 failures


...

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1416 - loss: 2.2851  


INFO :      fit progress: (2, 2.288743734359741, {'accuracy': 0.12999999523162842}, 11.988858707991312)
INFO :      configure_evaluate: strategy sampled 4 clients (out of 10)


Server-side evaluation round 2 with loss 2.288743734359741 / accuracy 0.12999999523162842
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0456 - loss: 2.3234  [repeated 5x across cluster]


INFO :      aggregate_evaluate: received 4 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 2 round(s) in 13.74s
INFO :      	History (loss, distributed):
INFO :      		round 1: 2.3146591186523438
INFO :      		round 2: 2.2906620502471924
INFO :      	History (loss, centralized):
INFO :      		round 0: 2.334395408630371
INFO :      		round 1: 2.30464506149292
INFO :      		round 2: 2.288743734359741
INFO :      	History (metrics, centralized):
INFO :      	{'accuracy': [(0, 0.09000000357627869),
INFO :      	              (1, 0.07999999821186066),
INFO :      	              (2, 0.12999999523162842)]}
INFO :      


(ClientAppActor pid=18566) [Client 1] evaluate, config: {} [repeated 7x across cluster]
1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.0938 - loss: 2.3022 [repeated 6x across cluster]
(ClientAppActor pid=18566) [Client 1] loss:2.2802670001983643, Client 1 accuracy:0.1899999976158142 [repeated 7x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0688 - loss: 2.3015  [repeated 3x across cluster]
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.0846 - loss: 2.3075 [repeated 2x across cluster]
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0554 - loss: 2.3007   [repeated 3x across cluster]


In [4]:
import pandas as pd

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Print the DataFrame
results_df

,Round,Loss,Accuracy
0,0,2.325810,0.11
1,1,2.309674,0.11
2,2,2.285895,0.12
3,0,2.309124,0.10
4,1,2.300936,0.13
5,2,2.282756,0.15
6,0,2.272108,0.15
7,1,2.246588,0.18
8,2,2.249062,0.10
9,0,2.334395,0.09


In [ ]:
# Print the results
for i, config in enumerate(configurations):
    print(f"Configuration {i+1}: fraction_fit={config['fraction_fit']}, fraction_evaluate={config['fraction_evaluate']}")
    for result in results:
        print(f"Round {result[0]}: Loss={result[1]}, Accuracy={result[2]}")

Configuration 1: fraction_fit=0.1, fraction_evaluate=0.1
Round 0: Loss=2.464465618133545, Accuracy=0.03999999910593033
Round 1: Loss=2.2994658946990967, Accuracy=0.11999999731779099
Round 2: Loss=2.392277240753174, Accuracy=0.09000000357627869
Round 3: Loss=2.3008837699890137, Accuracy=0.11999999731779099
Round 0: Loss=2.4440083503723145, Accuracy=0.05999999865889549
Round 1: Loss=2.30785870552063, Accuracy=0.14000000059604645
Round 2: Loss=2.3002302646636963, Accuracy=0.15000000596046448
Round 3: Loss=2.44368052482605, Accuracy=0.07999999821186066
Round 0: Loss=2.4666366577148438, Accuracy=0.10000000149011612
Round 1: Loss=2.3219473361968994, Accuracy=0.10999999940395355
Round 2: Loss=2.304999351501465, Accuracy=0.11999999731779099
Round 3: Loss=2.397759199142456, Accuracy=0.07999999821186066
Round 0: Loss=2.4729719161987305, Accuracy=0.10000000149011612
Round 1: Loss=2.3169243335723877, Accuracy=0.17000000178813934
Round 2: Loss=2.2722110748291016, Accuracy=0.18000000715255737
Round 

In [9]:
import pandas as pd

# Convert results to a DataFrame
results_df = pd.DataFrame(results, columns=['Round', 'Loss', 'Accuracy'])

# Print the DataFrame
results_df

,Round,Loss,Accuracy
0,0,2.464466,0.04
1,1,2.299466,0.12
2,2,2.392277,0.09
3,3,2.300884,0.12
4,0,2.444008,0.06
5,1,2.307859,0.14
6,2,2.300230,0.15
7,3,2.443681,0.08
8,0,2.466637,0.10
9,1,2.321947,0.11
